In [ ]:
%%configure -f
{"vCores": 16, "defaultLakehouse": {"name": "<YOUR_LAKEHOUSE_NAME>", "id": "<YOUR_LAKEHOUSE_ID>", "workspaceId": "<YOUR_WORKSPACE_ID>"}}


# fabric-rlm invoice-processing direct-vs-RLM evaluation

Fabric-native adaptation of `Trampoline-AI/predict-rlm/examples/invoice_processing`.
Downloads two small invoice PDFs (Acme Corporation + GlobalTech Solutions) — the
very PDFs predict-rlm ships in `sample/input/` — and asks the model to extract
structured invoice fields. Compares **direct single-call** vs **fabric_rlm.RLM**
coverage against ground-truth fields parsed from the reference
`invoice_extraction.xlsx` shipped with the example.

| Mode | Trigger | LM | Output root |
|---|---|---|---|
| **Fabric** | `/lakehouse/default` exists | `FabricChatLM` (notebook identity, no key) | `/lakehouse/default/Files/fabric_rlm_invoice_processing/<run_id>/` |
| **Local** | otherwise | `fabric_rlm.OpenAILM` (`OPENAI_API_KEY`) | `./_local_runs/invoice_processing/<run_id>/` |

Scoring rewards exact-field matches per invoice (vendor name substring, invoice
number, dates, subtotal/tax/total within 1% tolerance) plus line-item keyword
coverage. The line-item discount handling on the GlobalTech invoice is the
most diagnostic test of the skill.


In [ ]:
INV_RUN_ID = ''
INV_MODEL = 'gpt-5'
INV_MAX_TURNS = 20
INV_DIRECT_TEXT_CHARS = 40000
INV_PDF_URLS = [
    'https://raw.githubusercontent.com/Trampoline-AI/predict-rlm/2d93675d6d69b45f9eda9b8fc01e178323f8e6cb/examples/invoice_processing/sample/input/acme-invoice-2025-0042.pdf',
    'https://raw.githubusercontent.com/Trampoline-AI/predict-rlm/2d93675d6d69b45f9eda9b8fc01e178323f8e6cb/examples/invoice_processing/sample/input/globaltech-invoice-GT-10587.pdf',
]
INV_TIMEOUT_SECONDS = 1200


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys, time, hashlib, shutil, traceback, uuid

LAKEHOUSE_ROOT = Path('/lakehouse/default')
FABRIC_RUNTIME = LAKEHOUSE_ROOT.exists() and (LAKEHOUSE_ROOT / 'Files').exists()

RUN_ID = str(globals().get('INV_RUN_ID') or '').strip() or (
    time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8]
)

if FABRIC_RUNTIME:
    FILES_ROOT = LAKEHOUSE_ROOT / 'Files'
    RUN_ROOT = FILES_ROOT / 'fabric_rlm_invoice_processing' / RUN_ID
else:
    FILES_ROOT = Path.cwd() / '_local_runs'
    RUN_ROOT = FILES_ROOT / 'invoice_processing' / RUN_ID

INPUT_ROOT = RUN_ROOT / 'input'
DIRECT_ROOT = RUN_ROOT / 'direct'
RLM_ROOT = RUN_ROOT / 'rlm'
TRAJECTORY_ROOT = RUN_ROOT / 'trajectories'
for folder in [RUN_ROOT, INPUT_ROOT, DIRECT_ROOT, RLM_ROOT, TRAJECTORY_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

STAGE_EVENTS_PATH = RUN_ROOT / 'stage_events.jsonl'

def write_stage(stage, **details):
    payload = {'ts': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'stage': stage, **details}
    with open(STAGE_EVENTS_PATH, 'a', encoding='utf-8') as handle:
        handle.write(json.dumps(payload, ensure_ascii=False, default=str) + '\n')
        handle.flush()
    print(f'STAGE {stage}: {details}')

def write_json(relative_path, payload):
    path = RUN_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    return path

write_stage('bootstrap_start', run_root=str(RUN_ROOT), runtime='fabric' if FABRIC_RUNTIME else 'local')
print(f'Invoice run root: {RUN_ROOT}')

EXPECTED_SKILLS = ['pdf_document_analysis', 'validation', 'error_handling']

def _imports_ok():
    try:
        import fabric_rlm  # noqa: F401
        from fabric_rlm import File, RLM  # noqa: F401
        import fitz  # noqa: F401
        import nest_asyncio  # noqa: F401
        return True
    except Exception as exc:
        write_stage('import_probe_failed', error=repr(exc))
        return False

if FABRIC_RUNTIME:
    FABRIC_RLM_VERSION = os.environ.get('FABRIC_RLM_VERSION', '0.2.5')
    ISOLATED_DEPS_TARGET = Path(os.environ.get('FABRIC_RLM_INV_DEPS_TARGET', str(FILES_ROOT / 'fabric_rlm_invoice_processing' / '_deps' / ('fabric_rlm-' + FABRIC_RLM_VERSION + '-inv'))))

    def _prepend_path(p):
        s = str(p)
        if s not in sys.path:
            sys.path.insert(0, s)
        existing = os.environ.get('PYTHONPATH', '')
        if not existing or existing.split(os.pathsep)[0] != s:
            os.environ['PYTHONPATH'] = s + (os.pathsep + existing if existing else '')

    def _clear_modules(prefixes):
        for name in list(sys.modules):
            if any(name == p or name.startswith(p + '.') for p in prefixes):
                del sys.modules[name]

    if not ISOLATED_DEPS_TARGET.exists():
        ISOLATED_DEPS_TARGET.mkdir(parents=True, exist_ok=True)
        write_stage('bootstrap_install_start', package=f'fabric-rlm=={FABRIC_RLM_VERSION}', target=str(ISOLATED_DEPS_TARGET))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall',
                               '--index-url', 'https://test.pypi.org/simple', '--extra-index-url', 'https://pypi.org/simple',
                               '--target', str(ISOLATED_DEPS_TARGET), f'fabric-rlm=={FABRIC_RLM_VERSION}'])
    if ISOLATED_DEPS_TARGET.exists():
        _prepend_path(ISOLATED_DEPS_TARGET)
        _clear_modules(['fabric_rlm'])

    if not _imports_ok():
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
                               '--target', str(ISOLATED_DEPS_TARGET), 'pymupdf>=1.24.0,<1.25', 'nest_asyncio>=1.6'])
        _prepend_path(ISOLATED_DEPS_TARGET)
        _clear_modules(['fitz', 'pymupdf'])

    if not _imports_ok():
        raise ImportError('fabric_rlm/pymupdf unavailable after Fabric bootstrap')
else:
    if not _imports_ok():
        print('Local imports missing — attempting `pip install -e . pymupdf` ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(Path.cwd()), 'pymupdf>=1.24.0', 'nest_asyncio>=1.6'])
        for k in list(sys.modules):
            if k == 'fabric_rlm' or k.startswith('fabric_rlm.') or k in {'fitz', 'pymupdf'}:
                del sys.modules[k]
        if not _imports_ok():
            raise ImportError('fabric_rlm/pymupdf unavailable after local bootstrap')

import fabric_rlm
available = set(fabric_rlm.list_skills())
missing = set(EXPECTED_SKILLS) - available
if missing:
    raise RuntimeError(f'fabric_rlm at {fabric_rlm.__file__} missing skills: {sorted(missing)}; available={sorted(available)}')
write_stage('bootstrap_ok', fabric_rlm=str(fabric_rlm.__file__), version=getattr(fabric_rlm, '__version__', '?'),
            available_skills=sorted(available), runtime='fabric' if FABRIC_RUNTIME else 'local')


In [ ]:
import re, urllib.error, urllib.request
import fitz
import fabric_rlm
from fabric_rlm import File, RLM

MODEL = str(globals().get('INV_MODEL') or 'gpt-5')
MAX_TURNS = int(globals().get('INV_MAX_TURNS') or 20)
DIRECT_TEXT_CHARS = int(globals().get('INV_DIRECT_TEXT_CHARS') or 40000)
PDF_URLS = list(globals().get('INV_PDF_URLS') or [])
TIMEOUT_SECONDS = int(globals().get('INV_TIMEOUT_SECONDS') or 1200)
MODEL_KWARGS = {'max_tokens': 16000, 'temperature': 1.0}
_model_short = MODEL.split('/')[-1].lower()
_is_reasoning = _model_short.startswith(('gpt-5', 'o1', 'o3', 'o4')) and not _model_short.startswith('gpt-5-chat')
if _is_reasoning:
    MODEL_KWARGS = {'max_tokens': 32000, 'reasoning_effort': 'low'}

if len(PDF_URLS) < 1:
    raise ValueError('INV_PDF_URLS must list at least one invoice PDF URL.')

OUTPUT_SCHEMA_HINT = '''
Return an object with these keys:
  - invoices (list of objects, ONE per input PDF, in same order, with keys:
      vendor_name (str),
      invoice_number (str),
      date (str, ISO YYYY-MM-DD),
      due_date (str, ISO YYYY-MM-DD),
      subtotal (number, dollars),
      tax (number, dollars),
      total (number, dollars),
      line_items (list of {description, quantity, unit_price, amount}))
  - total_amount (number, sum of per-invoice totals, in dollars)
  - summary (string, one-paragraph overview of what was processed)
'''.strip()

CRITERIA = '''
Extract structured data from PDF invoices.
1. Survey the invoices — file names, page counts, vendor names visible on each.
2. For each invoice, extract: vendor_name, invoice_number, date, due_date,
   subtotal, tax, total, and every line_item with (description, quantity,
   unit_price, amount).
3. Preserve negative line items (e.g. discounts) with their negative amounts —
   do NOT omit them; the totals only reconcile when discounts are present.
4. Compute total_amount as the sum of per-invoice totals.
'''.strip()

# Ground-truth fields parsed from the reference invoice_extraction.xlsx
# shipped with the predict-rlm example.
GROUND_TRUTH = {
    'invoices': [
        {
            'vendor_substr': 'acme corporation',
            'invoice_number': 'INV-2025-0042',
            'date': '2025-03-15',
            'due_date': '2025-04-14',
            'subtotal': 3774.97,
            'tax': 311.43,
            'total': 4086.40,
            'line_item_keywords': [
                'cloud hosting', 'additional storage', 'premium support',
                'api calls', 'ssl certificate',
            ],
            'line_item_count': 5,
        },
        {
            'vendor_substr': 'globaltech',
            'invoice_number': 'GT-10587',
            'date': '2025-02-28',
            'due_date': '2025-03-30',
            'subtotal': 31000.00,
            'tax': 2817.90,
            'total': 30717.90,
            'line_item_keywords': [
                'software license', 'implementation', 'data migration',
                'user training', 'priority support', 'custom integration',
                'discount',
            ],
            'line_item_count': 7,
        },
    ],
    # 14 field anchors + 12 line-item keywords + 2 line-item-count anchors = 28
}

if FABRIC_RUNTIME:
    class FabricChatLM:
        def __init__(self, model, timeout=360, **default_kwargs):
            from synapse.ml.fabric.service_discovery import get_fabric_env_config
            from synapse.ml.fabric.token_utils import TokenUtils
            env = get_fabric_env_config().fabric_env_config
            base = f'{env.ml_workload_endpoint}cognitive/openai'.rstrip('/')
            self.model = model
            self.timeout = timeout
            self.default_kwargs = dict(default_kwargs)
            self.headers = {'Authorization': TokenUtils().get_openai_auth_header(), 'Content-Type': 'application/json'}
            self.urls = [
                f'{base}/openai/deployments/{model}/chat/completions?api-version=2025-04-01-preview',
                f'{base}/deployments/{model}/chat/completions?api-version=2025-04-01-preview',
            ]
        def __call__(self, *, messages, **kwargs):
            ck = {**self.default_kwargs, **kwargs}
            body = {'messages': messages}
            if ck.get('temperature') is not None:
                body['temperature'] = ck['temperature']
            if ck.get('max_tokens') is not None:
                body['max_completion_tokens'] = int(ck['max_tokens'])
            data = json.dumps(body).encode('utf-8')
            errors = []
            for i, url in enumerate(self.urls):
                req = urllib.request.Request(url, data=data, headers=self.headers, method='POST')
                try:
                    with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                        payload = json.loads(resp.read().decode('utf-8'))
                    choice = (payload.get('choices') or [{}])[0]
                    msg = choice.get('message') or {}
                    return {'content': msg.get('content') or choice.get('text') or '',
                            'usage': payload.get('usage') or {}, 'model': payload.get('model') or self.model}
                except urllib.error.HTTPError as exc:
                    detail = exc.read().decode('utf-8', errors='replace')[:2000]
                    errors.append({'url_index': i, 'status': exc.code, 'detail': detail})
                    if exc.code not in {400, 404}:
                        break
                except Exception as exc:
                    errors.append({'url_index': i, 'error': repr(exc)})
                    break
            raise RuntimeError(f'FabricChatLM call failed: {errors}')

    def make_direct_lm():
        return FabricChatLM(MODEL, timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    def make_rlm_lm():
        return FabricChatLM(MODEL, timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    SUB_LM_SPEC = 'fabric/' + MODEL
else:
    def _detect_local_provider():
        if os.environ.get('OPENAI_API_KEY'): return 'openai'
        if os.environ.get('OPENROUTER_API_KEY'): return 'openrouter'
        raise RuntimeError('Local runtime requires OPENAI_API_KEY or OPENROUTER_API_KEY.')
    def _local_model_spec():
        provider = _detect_local_provider()
        if provider == 'openai': return MODEL
        if MODEL.startswith('openrouter/'): return MODEL
        if '/' in MODEL: return 'openrouter/' + MODEL
        return 'openrouter/openai/' + MODEL
    def _make_local_lm(**extra_kwargs):
        import dspy
        provider = _detect_local_provider()
        kwargs = dict(MODEL_KWARGS); kwargs.update(extra_kwargs)
        if provider == 'openai':
            from fabric_rlm import OpenAILM
            return OpenAILM(MODEL, **kwargs)
        return dspy.LM(model=_local_model_spec(),
                       api_key=os.environ['OPENROUTER_API_KEY'],
                       api_base='https://openrouter.ai/api/v1', **kwargs)
    class _LocalDirectLM:
        def __init__(self, timeout=360, **default_kwargs):
            self.model = _local_model_spec(); self.timeout = timeout
            self._lm = _make_local_lm(**default_kwargs)
        def __call__(self, *, messages, **_kwargs):
            out = self._lm(messages=messages)
            if isinstance(out, list) and out:
                first = out[0]
                if isinstance(first, str): return {'content': first, 'usage': {}, 'model': self.model}
                if isinstance(first, dict): return {'content': first.get('content', ''), 'usage': {}, 'model': self.model}
            if isinstance(out, str): return {'content': out, 'usage': {}, 'model': self.model}
            return {'content': str(out), 'usage': {}, 'model': self.model}
    def make_direct_lm():
        _detect_local_provider(); return _LocalDirectLM(timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    def make_rlm_lm():
        return _make_local_lm()
    try: SUB_LM_SPEC = _local_model_spec()
    except Exception: SUB_LM_SPEC = ('openai/' + MODEL) if '/' not in MODEL else MODEL

def response_to_text(response):
    if isinstance(response, str): return response
    if isinstance(response, dict): return str(response.get('content') or response.get('text') or response)
    return str(getattr(response, 'content', response))

def extract_json_object(text):
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        return json.loads(cleaned)
    except Exception:
        start = cleaned.find('{'); end = cleaned.rfind('}')
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end + 1])
        raise

def normalize_items(value):
    if value is None: return []
    if isinstance(value, list): return value
    return [value]

def _to_float(x):
    if x is None: return None
    if isinstance(x, (int, float)): return float(x)
    try: return float(str(x).replace('$', '').replace(',', '').strip())
    except Exception: return None

def _float_match(actual, expected, rel=0.01, abs_tol=0.05):
    if actual is None or expected is None: return False
    try:
        a = float(actual); e = float(expected)
        return abs(a - e) <= max(abs_tol, abs(e) * rel)
    except Exception:
        return False

def score_invoices(analysis):
    '''Per-invoice field-level scoring against GROUND_TRUTH.

    Each invoice contributes (up to) the following anchors:
      - vendor_substr, invoice_number, date, due_date,
        subtotal, tax, total                                  = 7 field anchors
      - len(line_item_keywords) keywords (substring in any description)
      - line_item_count exact (or +/- 1)                      = 1 anchor
    Total = 14 (fields) + 12 (keywords) + 2 (counts) = 28.
    '''
    invoices = normalize_items(analysis.get('invoices'))
    invoices_by_index = {i: (invoices[i] if i < len(invoices) and isinstance(invoices[i], dict) else {})
                         for i in range(len(GROUND_TRUTH['invoices']))}

    groups = {}
    total_hits = 0
    total_possible = 0
    per_invoice_results = []

    for i, gt in enumerate(GROUND_TRUTH['invoices']):
        inv = invoices_by_index.get(i, {})
        field_hits = []
        field_misses = []

        vendor = str(inv.get('vendor_name', '')).lower()
        if gt['vendor_substr'] in vendor: field_hits.append('vendor_name')
        else: field_misses.append('vendor_name')

        inv_no = str(inv.get('invoice_number', '')).strip()
        if gt['invoice_number'].lower().replace('-', '') in inv_no.lower().replace('-', ''):
            field_hits.append('invoice_number')
        else: field_misses.append('invoice_number')

        for date_field in ('date', 'due_date'):
            actual = str(inv.get(date_field, '')).strip()[:10]
            if actual == gt[date_field]: field_hits.append(date_field)
            else: field_misses.append(date_field)

        for num_field in ('subtotal', 'tax', 'total'):
            if _float_match(_to_float(inv.get(num_field)), gt[num_field]):
                field_hits.append(num_field)
            else:
                field_misses.append(num_field)

        line_items = normalize_items(inv.get('line_items'))
        descriptions = ' || '.join(str(li.get('description', '')).lower() for li in line_items if isinstance(li, dict))
        kw_hits = [kw for kw in gt['line_item_keywords'] if kw in descriptions]
        kw_misses = [kw for kw in gt['line_item_keywords'] if kw not in descriptions]

        count_hit = len(line_items) == gt['line_item_count']
        count_close = abs(len(line_items) - gt['line_item_count']) <= 1
        count_anchor_hit = count_hit or count_close

        invoice_total_anchors = 7 + len(gt['line_item_keywords']) + 1
        invoice_hits = len(field_hits) + len(kw_hits) + (1 if count_anchor_hit else 0)
        total_hits += invoice_hits
        total_possible += invoice_total_anchors

        per_invoice_results.append({
            'index': i,
            'expected_vendor': gt['vendor_substr'],
            'actual_vendor': inv.get('vendor_name'),
            'field_hits': field_hits,
            'field_misses': field_misses,
            'line_item_keyword_hits': kw_hits,
            'line_item_keyword_misses': kw_misses,
            'line_item_count_actual': len(line_items),
            'line_item_count_expected': gt['line_item_count'],
            'line_item_count_match_exact': count_hit,
            'line_item_count_match_within_1': count_close,
            'invoice_score': invoice_hits,
            'invoice_possible': invoice_total_anchors,
        })

    # Roll-up groups for top-level reporting.
    groups = {
        'fields': {
            'score': sum(len(r['field_hits']) for r in per_invoice_results),
            'possible': 7 * len(GROUND_TRUTH['invoices']),
        },
        'line_item_keywords': {
            'score': sum(len(r['line_item_keyword_hits']) for r in per_invoice_results),
            'possible': sum(len(gt['line_item_keywords']) for gt in GROUND_TRUTH['invoices']),
        },
        'line_item_counts': {
            'score': sum(1 for r in per_invoice_results if r['line_item_count_match_within_1']),
            'possible': len(GROUND_TRUTH['invoices']),
        },
    }

    # total_amount anchor (1 extra, makes total_possible = 28+1=29)
    gt_total = sum(gt['total'] for gt in GROUND_TRUTH['invoices'])
    total_amount_hit = _float_match(_to_float(analysis.get('total_amount')), gt_total)
    if total_amount_hit:
        total_hits += 1
    total_possible += 1
    groups['total_amount'] = {'score': 1 if total_amount_hit else 0, 'possible': 1}

    return {
        'score': total_hits, 'possible': total_possible,
        'coverage': total_hits / max(total_possible, 1),
        'groups': groups,
        'per_invoice': per_invoice_results,
        'invoice_count_actual': len(invoices),
        'invoice_count_expected': len(GROUND_TRUTH['invoices']),
        'total_amount_actual': analysis.get('total_amount'),
        'total_amount_expected': gt_total,
        'total_amount_match': total_amount_hit,
    }


In [ ]:
pdf_paths = []
for url in PDF_URLS:
    name = url.rsplit('/', 1)[-1]
    target = INPUT_ROOT / name
    if not target.exists():
        write_stage('download_pdf_start', url=url, path=str(target))
        req = urllib.request.Request(url, headers={'User-Agent': 'fabric-rlm-invoice-processing'})
        with urllib.request.urlopen(req, timeout=180) as resp:
            target.write_bytes(resp.read())
        write_stage('download_pdf_done', bytes=target.stat().st_size, path=str(target))
    else:
        write_stage('download_pdf_cached', path=str(target), bytes=target.stat().st_size)
    pdf_paths.append(target)

pdf_text_blocks = []
pdf_metadata = []
for p in pdf_paths:
    doc = fitz.open(p)
    pages = [{'page': i + 1, 'text': page.get_text('text')} for i, page in enumerate(doc)]
    doc.close()
    full_text = '\n\n'.join(f'--- Page {row["page"]} ---\n{row["text"]}' for row in pages)
    text_path = INPUT_ROOT / (p.stem + '.txt')
    text_path.write_text(full_text, encoding='utf-8')
    excerpt = full_text[:DIRECT_TEXT_CHARS]
    pdf_text_blocks.append((p.name, excerpt))
    pdf_metadata.append({
        'name': p.name, 'pdf_path': str(p), 'text_path': str(text_path),
        'page_count': len(pages), 'text_chars': len(full_text),
        'direct_excerpt_chars': len(excerpt),
    })

manifest = {
    'run_id': RUN_ID, 'runtime': 'fabric' if FABRIC_RUNTIME else 'local',
    'source': 'Trampoline-AI/predict-rlm examples/invoice_processing sample PDFs',
    'pdf_urls': PDF_URLS,
    'documents': pdf_metadata,
    'model': MODEL, 'max_turns': MAX_TURNS,
    'fabric_rlm_version': getattr(fabric_rlm, '__version__', 'unknown'),
    'python': sys.version, 'platform': platform.platform(),
}
write_json('manifest.json', manifest)
write_stage('documents_ready',
            docs=[{'name': m['name'], 'pages': m['page_count'], 'text_chars': m['text_chars']}
                  for m in pdf_metadata])


In [ ]:
direct_lm = make_direct_lm()
direct_user_parts = [CRITERIA, OUTPUT_SCHEMA_HINT,
                     'You are receiving the extracted text of one or more invoice PDFs to extract. '
                     'This is a single-pass direct baseline; use only this text.']
for name, excerpt in pdf_text_blocks:
    direct_user_parts.append(f'\n=== INVOICE: {name} ===\n{excerpt}')
direct_messages = [
    {'role': 'system',
     'content': ('You extract structured invoice data and return ONLY valid JSON with keys '
                 'invoices, total_amount, summary. invoices is a list of objects per the schema. '
                 'Preserve negative line items like discounts.')},
    {'role': 'user', 'content': '\n\n'.join(direct_user_parts)},
]
write_stage('direct_start', model=MODEL,
            total_text_chars=sum(len(e) for _, e in pdf_text_blocks))
direct_started = time.perf_counter()
direct_response = direct_lm(messages=direct_messages)
direct_duration = time.perf_counter() - direct_started
direct_text = response_to_text(direct_response)
(DIRECT_ROOT / 'raw_response.txt').write_text(direct_text, encoding='utf-8')
try:
    direct_analysis = extract_json_object(direct_text)
    direct_error = None
except Exception as exc:
    direct_analysis = {'invoices': [], 'total_amount': None, 'summary': direct_text[:500]}
    direct_error = repr(exc)
direct_score = score_invoices(direct_analysis)
write_json('direct/analysis.json', direct_analysis)
write_json('direct/score.json', direct_score)
write_stage('direct_done', duration_s=direct_duration,
            score=direct_score['score'], possible=direct_score['possible'],
            coverage=round(direct_score['coverage'], 3),
            invoice_count=direct_score['invoice_count_actual'],
            total_amount_match=direct_score['total_amount_match'],
            parse_error=direct_error)
print(f"DIRECT  : score={direct_score['score']}/{direct_score['possible']}  "
      f"coverage={direct_score['coverage']:.0%}  "
      f"invoices={direct_score['invoice_count_actual']}  "
      f"total_amount_match={direct_score['total_amount_match']}  "
      f"in {direct_duration:.1f}s")


In [ ]:
RLM_TASK = '''
Extract structured invoice data from the provided PDF invoices.

You have access to the preloaded `pdf_document_analysis` skill — follow it for PDF work:
- Use Python and PyMuPDF (`fitz`) to open each PDF and record `page_count`.
- For each invoice, render each page as an image at ~200 DPI to a data URI and use
  `predict()` to extract: vendor_name, invoice_number, date, due_date, subtotal,
  tax, total, and ALL line_items (description, quantity, unit_price, amount).
- Use raw PDF text as a cross-check for the field values (numbers and IDs may be
  OCRed wrong from the image; confirm against text where possible).
- Use `asyncio.gather()` over independent `await predict(...)` calls per invoice.
- PRESERVE negative line items (discounts, credits) — do not silently drop them.
  The reported subtotal/tax/total only reconcile when discounts are included.
- ALL date fields must be ISO format YYYY-MM-DD.

Write a JSON analysis to {output_dir}/analysis.json.

Before SUBMIT, run a self-check and repair any failure:
- One invoice object per input PDF, in the same order as the input list.
- For each invoice: vendor_name non-empty, invoice_number non-empty, dates ISO,
  and (sum of line_item amounts) reconciles with subtotal (within $0.10).
- subtotal + tax = total (within $0.10), unless a discount line item is present.
- total_amount = sum of per-invoice totals (within $0.10).
- Unsupported claims removed.

Call SUBMIT(invoices=..., total_amount=..., summary=..., analysis_path=...,
            page_counts=...).
Required JSON-friendly shapes:
- invoices: list of dicts with keys vendor_name, invoice_number, date, due_date,
  subtotal, tax, total, line_items.
- line_items: list of dicts with keys description, quantity, unit_price, amount
  (numbers as floats; negative amounts permitted for discounts).
- page_counts: dict mapping invoice filename to page_count.
'''.strip()

write_stage('rlm_setup_start', model=MODEL, max_turns=MAX_TURNS)
try:
    rlm_lm = make_rlm_lm()
    invoice_files = [File(str(p)) for p in pdf_paths]
    rlm = RLM.task(
        task=RLM_TASK,
        inputs={'invoices': invoice_files, 'criteria': CRITERIA, 'output_dir': str(RLM_ROOT)},
        outputs=['invoices', 'total_amount', 'summary', 'analysis_path', 'page_counts'],
        lm=rlm_lm,
        sub_lm=SUB_LM_SPEC,
        max_turns=MAX_TURNS,
        skills=['pdf_document_analysis'],
        enable_skill_autoloading=True,
        timeout=TIMEOUT_SECONDS,
    )
except Exception as exc:
    tb = traceback.format_exc()
    write_stage('rlm_setup_failed', error=repr(exc), traceback=tb[-4000:])
    (RLM_ROOT / 'setup_error.txt').write_text(tb, encoding='utf-8')
    raise
write_stage('rlm_start', model=MODEL, max_turns=MAX_TURNS, n_invoices=len(invoice_files))
rlm_started = time.perf_counter()
try:
    rlm_result = rlm.run()
except Exception as exc:
    tb = traceback.format_exc()
    write_stage('rlm_run_failed', error=repr(exc), traceback=tb[-4000:],
                duration_s=time.perf_counter() - rlm_started)
    (RLM_ROOT / 'run_error.txt').write_text(tb, encoding='utf-8')
    raise
rlm_duration = time.perf_counter() - rlm_started

trajectory_path = TRAJECTORY_ROOT / 'invoice_processing_rlm.jsonl'
trajectory_path.parent.mkdir(parents=True, exist_ok=True)
rlm_result.trajectory.write_jsonl(trajectory_path)

if rlm_result.submitted and rlm_result.payload:
    rlm_analysis = dict(rlm_result.payload)
else:
    rlm_analysis = {'invoices': [], 'total_amount': None, 'summary': '',
                    'failure_reason': rlm_result.failure_reason}

rlm_score = score_invoices(rlm_analysis)
write_json('rlm/analysis.json', rlm_analysis)
write_json('rlm/score.json', rlm_score)
write_stage('rlm_done', duration_s=rlm_duration, submitted=rlm_result.submitted,
            turns=len(rlm_result.trajectory.turns),
            score=rlm_score['score'], possible=rlm_score['possible'],
            coverage=round(rlm_score['coverage'], 3),
            invoice_count=rlm_score['invoice_count_actual'],
            total_amount_match=rlm_score['total_amount_match'])
print(f"RLM     : submitted={rlm_result.submitted}  turns={len(rlm_result.trajectory.turns)}  "
      f"score={rlm_score['score']}/{rlm_score['possible']}  "
      f"coverage={rlm_score['coverage']:.0%}  "
      f"invoices={rlm_score['invoice_count_actual']}  "
      f"total_amount_match={rlm_score['total_amount_match']}  "
      f"in {rlm_duration:.1f}s")


In [ ]:
record = {
    'run_id': RUN_ID, 'runtime': 'fabric' if FABRIC_RUNTIME else 'local',
    'model': MODEL,
    'source_example': 'https://github.com/Trampoline-AI/predict-rlm/tree/main/examples/invoice_processing',
    'documents': [m['name'] for m in pdf_metadata],
    'direct': {'duration_s': direct_duration, 'score': direct_score, 'parse_error': direct_error},
    'rlm':    {'duration_s': rlm_duration,    'submitted': rlm_result.submitted,
               'failure_reason': rlm_result.failure_reason, 'score': rlm_score,
               'turns': len(rlm_result.trajectory.turns),
               'trajectory_path': str(trajectory_path)},
}
record['rlm_upgrade'] = (
    rlm_score['score'] > direct_score['score']
    or rlm_score['coverage'] > direct_score['coverage']
)
write_json('record.json', record)

print()
print('=' * 70)
print('INVOICE PROCESSING — DIRECT vs RLM')
print('=' * 70)
for label, sc, dur in [('DIRECT', direct_score, direct_duration),
                       ('RLM   ', rlm_score, rlm_duration)]:
    print(f'{label}:  coverage={sc["coverage"]:.0%}  '
          f'({sc["score"]}/{sc["possible"]} anchors)  '
          f'invoices={sc["invoice_count_actual"]}  '
          f'total_amount={sc["total_amount_actual"]}  match={sc["total_amount_match"]}  '
          f'in {dur:.1f}s')
print()
print('Per-anchor-group hit rate:')
for group in ['fields', 'line_item_keywords', 'line_item_counts', 'total_amount']:
    d = direct_score['groups'].get(group, {'score': 0, 'possible': 0})
    r = rlm_score['groups'].get(group, {'score': 0, 'possible': 0})
    print(f'  {group:22s}  direct {d["score"]}/{d["possible"]}   rlm {r["score"]}/{r["possible"]}')
print()
print('Per-invoice details:')
for direct_inv, rlm_inv in zip(direct_score['per_invoice'], rlm_score['per_invoice']):
    print(f'  invoice[{direct_inv["index"]}] expected vendor: {direct_inv["expected_vendor"]}')
    print(f'    DIRECT: {direct_inv["invoice_score"]}/{direct_inv["invoice_possible"]}  '
          f'fields hit: {direct_inv["field_hits"]}  line items: {direct_inv["line_item_count_actual"]}/{direct_inv["line_item_count_expected"]}')
    print(f'    RLM   : {rlm_inv["invoice_score"]}/{rlm_inv["invoice_possible"]}  '
          f'fields hit: {rlm_inv["field_hits"]}  line items: {rlm_inv["line_item_count_actual"]}/{rlm_inv["line_item_count_expected"]}')
print()
print(f'Run root: {RUN_ROOT}')
print(f'Trajectory: {trajectory_path}')
print(f'RLM upgraded over direct? {record["rlm_upgrade"]}')
